# 🛡️ European Cyber Threat Collector

**Data source:** [AbuseIPDB](https://www.abuseipdb.com)  
**Coverage:** European countries only  
**Free tier:** 1,000 requests/day · resets at midnight UTC (1:00 AM Prague / CET)

> ⚠️ Raw IP data must not be publicly redistributed according to AbuseIPDB Terms of Service.

---
## Install & Import Libraries

In [1]:
import sys
import requests
import pandas as pd
import os
import time
from tqdm.auto import tqdm
from datetime import datetime, timezone

print('✅ Libraries ready')

✅ Libraries ready


---
## Configuration

Before running the project, add your API key.

1. Register for a free account at AbuseIPDB
2. Navigate to Account > API > Create Key
3. Copy your API key and paste it into the configuration section of the script.

✏️ Note: The free tier allows 1,000 API requests per day.

In [ ]:
# Paste your API key here
ABUSEIPDB_API_KEY = os.environ.get("ABUSEIPDB_API_KEY", "ASTE_YOUR_KEY_HERE")

# Settings
DAYS_BACK      = 365   # max 365 on free tier
MIN_CONFIDENCE = 75    # minimum abuse confidence score (0-100)
OUTPUT_FILE    = os.path.join(os.getcwd(), "abuseipdb_europe.csv")

# European country codes
EUROPEAN_COUNTRIES = {
    "AL","AD","AT","BY","BE","BA","BG","HR","CY","CZ","DK","EE","FI",
    "FR","DE","GR","HU","IS","IE","IT","XK","LV","LI","LT","LU","MT",
    "MD","MC","ME","NL","MK","NO","PL","PT","RO","RU","SM","RS","SK",
    "SI","ES","SE","CH","UA","GB","VA"
}

COUNTRY_NAMES = {
    "AT":"Austria",    "BE":"Belgium",        "BG":"Bulgaria",    "HR":"Croatia",
    "CY":"Cyprus",     "CZ":"Czech Republic",  "DK":"Denmark",    "EE":"Estonia",
    "FI":"Finland",    "FR":"France",          "DE":"Germany",    "GR":"Greece",
    "HU":"Hungary",    "IE":"Ireland",         "IT":"Italy",      "LV":"Latvia",
    "LT":"Lithuania",  "LU":"Luxembourg",      "MT":"Malta",      "NL":"Netherlands",
    "PL":"Poland",     "PT":"Portugal",        "RO":"Romania",    "SK":"Slovakia",
    "SI":"Slovenia",   "ES":"Spain",           "SE":"Sweden",     "GB":"United Kingdom",
    "UA":"Ukraine",    "RU":"Russia",          "NO":"Norway",     "CH":"Switzerland",
    "IS":"Iceland",    "BA":"Bosnia",          "RS":"Serbia",     "ME":"Montenegro",
    "MK":"North Macedonia", "AL":"Albania",    "MD":"Moldova",    "BY":"Belarus",
    "XK":"Kosovo",     "AD":"Andorra",         "LI":"Liechtenstein", "MC":"Monaco",
    "SM":"San Marino", "VA":"Vatican",
}

ATTACK_CATEGORIES = {
     1:"DNS Compromise",  2:"DNS Poisoning",   3:"Fraud Orders",   4:"DDoS Attack",
     5:"FTP Brute-Force", 6:"Ping of Death",   7:"Phishing",       8:"Fraud VoIP",
     9:"Open Proxy",     10:"Web Spam",        11:"Email Spam",    12:"Blog Spam",
    13:"VPN IP",         14:"Port Scan",       15:"Hacking",       16:"SQL Injection",
    17:"Spoofing",       18:"Brute-Force",     19:"Bad Web Bot",   20:"Exploited Host",
    21:"Web App Attack", 22:"SSH Abuse",       23:"IoT Targeted",
}

if ABUSEIPDB_API_KEY == "PASTE_YOUR_KEY_HERE":
    print("❌ API key not set! Paste your key into ABUSEIPDB_API_KEY above")
else:
    print("✅ Configuration ready")
    print(f"   Output file : {OUTPUT_FILE}")
    print(f"   Days back   : {DAYS_BACK}")
    print(f"   Min score   : {MIN_CONFIDENCE}%")

✅ Configuration ready
   Output file : c:\Users\andre\Documents\Python_Scripts\cyber_threat_collector\abuseipdb_europe.csv
   Days back   : 365
   Min score   : 75%


---
## Check Daily API Quota

In [4]:
headers = {"Key": ABUSEIPDB_API_KEY, "Accept": "application/json"}

print("Checking API quota...")
try:
    test = requests.get(
        "https://api.abuseipdb.com/api/v2/check",
        headers=headers,
        params={"ipAddress": "8.8.8.8", "maxAgeInDays": 1},
        timeout=15,
    )
    if test.status_code == 401:
        print("❌ Invalid API key - check Cell 2")
        QUOTA_REMAINING = 0
    elif test.status_code == 429:
        print("❌ Daily limit reached. Resets at 1:00 AM Prague time.")
        QUOTA_REMAINING = 0
    else:
        QUOTA_REMAINING = int(test.headers.get("X-RateLimit-Remaining", 0))
        QUOTA_LIMIT     = int(test.headers.get("X-RateLimit-Limit", 1000))
        QUOTA_USED      = QUOTA_LIMIT - QUOTA_REMAINING
        MAX_EU_CHECKS   = max(0, QUOTA_REMAINING - 3)  # safety buffer

        print(f"✅ Quota status:")
        print(f"   Daily limit          : {QUOTA_LIMIT:,}")
        print(f"   Used today           : {QUOTA_USED:,}")
        print(f"   Remaining            : {QUOTA_REMAINING:,}")
        print(f"   Max EU IPs to enrich : {MAX_EU_CHECKS:,}")
        print()
        print("ℹ️ This version uses ip-api.com (free, no key) to geolocate all 10,000 IPs")
        print("   and only spends AbuseIPDB quota on European IPs - far more efficient!")

        if QUOTA_REMAINING < 10:
            print()
            print("⚠️ Not enough quota left. Come back after 1:00 AM Prague time.")

except Exception as e:
    print(f"❌ Error: {e}")
    QUOTA_REMAINING = 0
    MAX_EU_CHECKS   = 0

Checking API quota...
✅ Quota status:
   Daily limit          : 1,000
   Used today           : 1
   Remaining            : 999
   Max EU IPs to enrich : 996

ℹ️ This version uses ip-api.com (free, no key) to geolocate all 10,000 IPs
   and only spends AbuseIPDB quota on European IPs - far more efficient!


---
## Download Blacklist + Geolocate All IPs via ip-api.com

The script retrieves 10,000 reported malicious IP addresses from AbuseIPDB using a single API request.
It then performs bulk geolocation for all IPs via ip-api.com using the batch endpoint.

This step is completely free and does not require an API key.

After geolocation, the dataset is filtered to include European IP addresses only, reducing the list to the relevant subset for further analysis.

In [5]:
if QUOTA_REMAINING < 10:
    print("⚠️ Not enough quota. Run Cell 3 first.")
    european_candidates = []
else:
    # Download AbuseIPDB blacklist (1 request)
    print(f"[1/2] Downloading blacklist (confidence >= {MIN_CONFIDENCE}%, last {DAYS_BACK} days)...")
    try:
        resp = requests.get(
            "https://api.abuseipdb.com/api/v2/blacklist",
            headers=headers,
            params={
                "confidenceMinimum": MIN_CONFIDENCE,
                "limit"            : 10000,
                "maxAgeInDays"     : DAYS_BACK,
            },
            timeout=60,
        )
        resp.raise_for_status()
        blacklist = resp.json().get("data", [])
        print(f"   ✅ Got {len(blacklist):,} IPs from blacklist")
    except Exception as e:
        print(f"   ❌ Failed to download blacklist: {e}")
        blacklist = []

    # Geolocate all IPs via ip-api.com (free, no key)
    # ip-api allows 100 IPs per batch request, 15 requests/min on free tier
    print(f"\n[2/2]  Geolocating {len(blacklist):,} IPs via ip-api.com (free)...")
    print(      "      This uses no AbuseIPDB quota at all.")

    ip_to_country = {}   # {ip: country_code}
    batch_size    = 100
    all_ips       = [entry["ipAddress"] for entry in blacklist if entry.get("ipAddress")]

    for i in tqdm(range(0, len(all_ips), batch_size), desc="     Geolocating batches"):
        batch = all_ips[i:i + batch_size]
        try:
            geo_resp = requests.post(
                "http://ip-api.com/batch",
                json=[{"query": ip, "fields": "query,countryCode"} for ip in batch],
                timeout=15,
            )
            if geo_resp.status_code == 200:
                for item in geo_resp.json():
                    ip_to_country[item["query"]] = item.get("countryCode", "").upper()
            elif geo_resp.status_code == 429:
                print("   ⚠️  ip-api.com rate limit hit - waiting 60 seconds...")
                time.sleep(61)
                geo_resp = requests.post(
                    "http://ip-api.com/batch",
                    json=[{"query": ip, "fields": "query,countryCode"} for ip in batch],
                    timeout=15,
                )
                if geo_resp.status_code == 200:
                    for item in geo_resp.json():
                        ip_to_country[item["query"]] = item.get("countryCode", "").upper()
        except Exception:
            continue
        time.sleep(4.2)   # ip-api free: 15 requests/min = 1 per 4 seconds

    # Filter to European IPs
    european_candidates = [
        entry for entry in blacklist
        if ip_to_country.get(entry.get("ipAddress", ""), "") in EUROPEAN_COUNTRIES
    ]

    print(f"\n     ✅ Geolocation complete")
    print(f"   Total IPs geolocated  : {len(ip_to_country):,}")
    print(f"   European IPs found    : {len(european_candidates):,}")
    print(f"   AbuseIPDB quota used  : 1 request (blacklist download only)")

    if len(european_candidates) > MAX_EU_CHECKS:
        print()
        print(f"   ⚠️ {len(european_candidates):,} EU IPs found but only {MAX_EU_CHECKS:,} quota remaining.")
        print(f"       Will enrich first {MAX_EU_CHECKS:,} — run again tomorrow for the rest.")

[1/2] Downloading blacklist (confidence >= 75%, last 365 days)...
   ✅ Got 10,000 IPs from blacklist

[2/2]  Geolocating 10,000 IPs via ip-api.com (free)...
      This uses no AbuseIPDB quota at all.


     Geolocating batches:   0%|          | 0/100 [00:00<?, ?it/s]


     ✅ Geolocation complete
   Total IPs geolocated  : 10,000
   European IPs found    : 2,346
   AbuseIPDB quota used  : 1 request (blacklist download only)

   ⚠️ 2,346 EU IPs found but only 996 quota remaining.
       Will enrich first 996 — run again tomorrow for the rest.


---
## Enrich European IPs via AbuseIPDB /check

In this stage, the script uses the AbuseIPDB API `/check` endpoint to enrich only the previously filtered European IP addresses.

By limiting requests to European IPs, the project optimizes the daily API quota and avoids unnecessary checks.

The enrichment step retrieves additional threat intelligence, including:
- ISP (Internet Service Provider)
- TOR exit node status
- Attack categories
- Report statistics

The request uses ``verbose=True``, which provides detailed attack category information for each IP address.

In [6]:
if not european_candidates:
    print("⚠️ No European candidates — run Cell 4 first.")
    european_ips = []
else:
    to_enrich     = european_candidates[:MAX_EU_CHECKS]
    european_ips  = []
    requests_used = 0

    est_min = len(to_enrich) * 1.5 / 60
    print(f"Enriching {len(to_enrich):,} European IPs via AbuseIPDB /check...")
    print(f"Estimated time: {est_min:.0f}–{est_min * 1.3:.0f} minutes\n")

    for entry in tqdm(to_enrich, desc="Enriching EU IPs"):
        ip           = entry.get("ipAddress")
        country_code = ip_to_country.get(ip, "")
        if not ip:
            continue

        try:
            resp = requests.get(
                "https://api.abuseipdb.com/api/v2/check",
                headers=headers,
                params={
                    "ipAddress"   : ip,
                    "maxAgeInDays": DAYS_BACK,
                    "verbose"     : True,   # returns full report history with categories
                },
                timeout=15,
            )
            requests_used += 1

            if resp.status_code == 429:
                print(f"\n⚠️ Rate limit reached after {requests_used} requests.")
                print(f"   Saving {len(european_ips)} enriched IPs collected so far.")
                break

            if resp.status_code != 200:
                time.sleep(0.15)
                continue

            detail = resp.json().get("data", {})

            # Collect all category IDs across all reports, deduplicate, decode
            all_cat_ids = []
            for report in detail.get("reports", []):
                all_cat_ids.extend(report.get("categories", []))
            unique_cat_ids = sorted(set(all_cat_ids))
            cat_names      = [ATTACK_CATEGORIES.get(c, f"Category {c}") for c in unique_cat_ids]

            european_ips.append({
                "ip_address"       : ip,
                "country_name"     : COUNTRY_NAMES.get(country_code, country_code),
                "abuse_score"      : detail.get("abuseConfidenceScore"),
                "attack_categories": ", ".join(cat_names) if cat_names else "Unknown",
                "total_reports"    : detail.get("totalReports"),
                "isp"              : detail.get("isp", ""),
                "is_tor"           : detail.get("isTor", False),
                "last_reported_at" : detail.get("lastReportedAt"),
            })

            time.sleep(0.15)

        except requests.exceptions.Timeout:
            continue
        except Exception:
            continue

    print(f"\n✅ Enriched {len(european_ips):,} European IPs")
    print(f"    AbuseIPDB requests used : {requests_used + 1:,} (including blacklist)")
    print(f"    Remaining quota         : ~{max(0, QUOTA_REMAINING - requests_used):,}")

    # Quick preview of attack categories
    if european_ips:
        sample_cats = set()
        for row in european_ips:
            for cat in row["attack_categories"].split(", "):
                sample_cats.add(cat)
        sample_cats.discard("Unknown")
        if sample_cats:
            print(f"   Attack types found      : {', '.join(sorted(sample_cats)[:8])}")

Enriching 996 European IPs via AbuseIPDB /check...
Estimated time: 25–32 minutes



Enriching EU IPs:   0%|          | 0/996 [00:00<?, ?it/s]


✅ Enriched 995 European IPs
    AbuseIPDB requests used : 996 (including blacklist)
    Remaining quota         : ~4
   Attack types found      : Bad Web Bot, Blog Spam, Brute-Force, DDoS Attack, DNS Compromise, DNS Poisoning, Email Spam, Exploited Host


---
## Save to CSV & Print Summary

In [7]:
if not european_ips:
    print("⚠️ No data to save. Check that Cells 4 and 5 ran successfully.")
else:
    COLUMNS = [
        "ip_address", "country_name", "abuse_score",
        "attack_categories", "total_reports", "isp",
        "is_tor", "last_reported_at"
    ]

    new_df = pd.DataFrame(european_ips, columns=COLUMNS)
    new_df["last_reported_at"] = pd.to_datetime(
        new_df["last_reported_at"], errors="coerce", utc=True
    ).dt.date

    if os.path.exists(OUTPUT_FILE):
        existing_df = pd.read_csv(OUTPUT_FILE)
        existing_df["last_reported_at"] = pd.to_datetime(
            existing_df["last_reported_at"], errors="coerce"
        ).dt.date
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
        combined_df.drop_duplicates(
            subset=["ip_address", "last_reported_at"], keep="last", inplace=True
        )
        combined_df.sort_values(
            ["last_reported_at", "abuse_score"], ascending=[False, False], inplace=True
        )
        combined_df.reset_index(drop=True, inplace=True)
        combined_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
        mode_label = "Appended & deduplicated"
        df = combined_df
    else:
        new_df.sort_values(
            ["last_reported_at", "abuse_score"], ascending=[False, False], inplace=True
        )
        new_df.reset_index(drop=True, inplace=True)
        new_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
        mode_label = "Created"
        df = new_df

    print(f"💾 {mode_label} → {OUTPUT_FILE}")
    print(f"   New rows this run  : {len(new_df):,}")
    print(f"   Total rows in file : {len(df):,}")
    print()

    print("📋 Sample output (first 5 rows):")
    try:
        display(df.drop(columns=["ip_address"]).head())
    except NameError:
        print(df.drop(columns=["ip_address"]).head().to_string()) 

    scores = pd.to_numeric(df["abuse_score"], errors="coerce")
    dates  = pd.to_datetime(df["last_reported_at"], errors="coerce").dropna()

    print()
    print("=" * 50)
    print("  DATASET SUMMARY")
    print("=" * 50)
    print(f"  Total rows          : {len(df):,}")
    print(f"  Unique IPs          : {df['ip_address'].nunique():,}")
    print(f"  Countries           : {df['country_name'].nunique()}")
    print(f"  Unique ISPs         : {df['isp'].nunique():,}")
    print(f"  Avg abuse score     : {scores.mean():.1f} / 100")
    print(f"  TOR exit nodes      : {int(df['is_tor'].sum()):,}")

    if len(dates) > 0:
        print(f"  Date range          : {dates.min().date()} → {dates.max().date()}")
    else:
        print(f"  Date range          : N/A")

    print()
    print("  Abuse Score Breakdown:")
    print(f"    Critical (100)   : {(scores == 100).sum():,} IPs")
    print(f"    High     (90-99) : {((scores >= 90) & (scores < 100)).sum():,} IPs")
    print(f"    Medium   (75-89) : {((scores >= 75) & (scores < 90)).sum():,} IPs")
    print()
    print("  Top 5 Countries:")
    print(df["country_name"].value_counts().head(5).to_string())
    print()
    print("  Top 5 Attack Types:")
    top_attacks = (
        df["attack_categories"].dropna()
        .str.split(", ").explode().str.strip()
        .value_counts().head(5)
    )
    print(top_attacks.to_string())
    print()
    print("  Top 5 ISPs:")
    print(df["isp"].value_counts().head(5).to_string())
    print()
    print("=" * 50)
    print("  ✅ Done! Run again after 1:00 AM Prague time.")
    print("=" * 50)

💾 Appended & deduplicated → c:\Users\andre\Documents\Python_Scripts\cyber_threat_collector\abuseipdb_europe.csv
   New rows this run  : 995
   Total rows in file : 4,193

📋 Sample output (first 5 rows):


,collected_at,country_code,country_name,abuse_score,total_reports,distinct_reporters,last_reported_at,isp,domain,usage_type,hostnames,is_tor,attack_categories
0,NaN,NaN,Netherlands,100,25049,NaN,2026-03-18,TECHOFF SRV LIMITED,NaN,NaN,NaN,False,"DNS Compromise, DNS Poisoning, DDoS Attack, FT..."
1,NaN,NaN,Andorra,100,2017,NaN,2026-03-18,TECHOFF SRV LIMITED,NaN,NaN,NaN,False,"DNS Compromise, DNS Poisoning, DDoS Attack, FT..."
2,NaN,NaN,Netherlands,100,9591,NaN,2026-03-18,TECHOFF SRV LIMITED,NaN,NaN,NaN,False,"DNS Compromise, DNS Poisoning, DDoS Attack, FT..."
3,NaN,NaN,Ukraine,100,2867,NaN,2026-03-18,GMHOST datacenter,NaN,NaN,NaN,False,"DDoS Attack, FTP Brute-Force, Ping of Death, P..."
4,NaN,NaN,Sweden,100,5997,NaN,2026-03-18,Vaxjo NET C2IP,NaN,NaN,NaN,False,"DDoS Attack, FTP Brute-Force, Ping of Death, P..."



  DATASET SUMMARY
  Total rows          : 4,193
  Unique IPs          : 2,786
  Countries           : 36
  Unique ISPs         : 589
  Avg abuse score     : 100.0 / 100
  TOR exit nodes      : 22
  Date range          : 2026-03-14 → 2026-03-18

  Abuse Score Breakdown:
    Critical (100)   : 4,186 IPs
    High     (90-99) : 7 IPs
    Medium   (75-89) : 0 IPs

  Top 5 Countries:
country_name
Netherlands       874
France            768
United Kingdom    693
Germany           548
Russia            333

  Top 5 Attack Types:
attack_categories
Brute-Force       3921
Port Scan         3913
Hacking           3890
Web App Attack    3791
SSH Abuse         3780

  Top 5 ISPs:
isp
FR ONYPHE              322
Driftnet Ltd           315
DigitalOcean, LLC      271
TECHOFF SRV LIMITED    226
NL MODAT               136

  ✅ Done! Run again after 1:00 AM Prague time.
